# Creación del Datamart Analítico

Este notebook construye el datamart en esquema estrella a partir de las tablas operacionales
(`creditos`, `amortizacion`, `juicios`) cargadas por el ETL.

### Pasos

1. **Vista materializada `mv_creditos_mensuales`**: encapsula el JOIN de 3 tablas + agregación
   por mes-bloque + cálculo de tasas y crisis_flag. Es la **fuente de verdad única** para
   entrenamiento, predicción y BI.
2. **Dimensiones**: `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal` (upsert idempotente).
3. **Tabla de hechos**: `fact_creditos_mensual` poblada desde la MV con joins a dimensiones.
4. **Validación**: verificación de integridad y consistencia.

### Entradas
- Tablas: `creditos`, `amortizacion`, `juicios` en PostgreSQL.

### Salidas
- Vista materializada: `mv_creditos_mensuales`
- Dimensiones: `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal`
- Tabla de hechos: `fact_creditos_mensual`

> Librerias:
> !pip install polars psycopg2-binary

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.insert(0, '..')

import polars as pl
import psycopg2
from src.sql import (
    SQL_CREA_DIM_RIESGO,
    SQL_CREA_DIM_SECTOR,
    SQL_CREA_DIM_SUCURSAL,
    SQL_CREA_DIM_TIEMPO,
    SQL_CREA_FACT_CREDITOS,
    SQL_CREATE_IDX_MV,
    SQL_CREATE_MV_CREDITOS_MENSUALES,
    SQL_INSERT_DIM_RIESGO,
    SQL_INSERT_DIM_SECTOR,
    SQL_INSERT_DIM_SUCURSAL,
    SQL_INSERT_DIM_TIEMPO,
    SQL_UPSERT_FACT_CREDITOS,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "postgres_db",
    "user": "postgres_usr",
    "password": "admin123",
}

conn = psycopg2.connect(
    host=DB_CONFIG['host'],
    port=DB_CONFIG['port'],
    dbname=DB_CONFIG['database'],
    user=DB_CONFIG['user'],
    password=DB_CONFIG['password']
)
conn.autocommit = True

## Crear tablas de dimensiones y hechos (DDL)

Se crea el esquema estrella una sola vez. Las tablas se mantienen si ya existen.

In [53]:
ddl_dim_tiempo = SQL_CREA_DIM_TIEMPO
ddl_dim_riesgo = SQL_CREA_DIM_RIESGO
ddl_dim_sector = SQL_CREA_DIM_SECTOR
ddl_dim_sucursal = SQL_CREA_DIM_SUCURSAL
ddl_fact_creditos = SQL_CREA_FACT_CREDITOS

cursor = conn.cursor()
for nombre, ddl in [
    ("dim_tiempo", ddl_dim_tiempo),
    ("dim_riesgo", ddl_dim_riesgo),
    ("dim_sector", ddl_dim_sector),
    ("dim_sucursal", ddl_dim_sucursal),
    ("fact_creditos_mensual", ddl_fact_creditos),
]:
    cursor.execute(ddl)
    print(f"OK: {nombre}")
cursor.close()

OK: dim_tiempo
OK: dim_riesgo
OK: dim_sector
OK: dim_sucursal
OK: fact_creditos_mensual


## Crear vista materializada `mv_creditos_mensuales`

La vista materializada encapsula la lógica completa de:
- JOIN de las 3 tablas operacionales
- Agregación por mes-bloque (riesgo × sector × sucursal)
- Cálculo de tasas derivadas (mora_90, judicial, cierre, etc.)
- Cálculo del `crisis_flag` con 11 reglas de scoring
- Generación del `bloque_id`

Esta es la **fuente de verdad única** que reemplaza las queries duplicadas en EDA, train y predict.

In [ ]:
cursor = conn.cursor()
print("Creando vista materializada mv_creditos_mensuales...")
cursor.execute(SQL_CREATE_MV_CREDITOS_MENSUALES)

print("Creando indice unico para REFRESH CONCURRENTLY...")
cursor.execute(SQL_CREATE_IDX_MV)
cursor.close()

print("Vista materializada mv_creditos_mensuales creada.")

Eliminando vista materializada anterior (si existe)...
Creando vista materializada mv_creditos_mensuales...
Creando indice unico para REFRESH CONCURRENTLY...
Vista materializada mv_creditos_mensuales creada.


### Verificar la vista materializada

In [55]:
df_mv = pl.read_database("SELECT COUNT(*) AS total FROM mv_creditos_mensuales", conn)
print(f"Registros en mv_creditos_mensuales: {df_mv['total'][0]:,}")

df_sample = pl.read_database(
    "SELECT * FROM mv_creditos_mensuales ORDER BY mes LIMIT 5",
    conn
)
print("\nMuestra:")
df_sample

Registros en mv_creditos_mensuales: 5,562

Muestra:


mes,riesgo,sector,codigo_sucursal,num_creditos,monto_total,monto_promedio,plazo_promedio,tasa_interes_promedio,saldo_promedio,total_costo_judicial,total_gestion_cobro,total_notificaciones,tot_dias_mora_promedio,tot_num_moras_promedio,mora_promedio,creditos_judiciales,creditos_cerrados,num_clientes_unicos,tasa_judicial,tasa_cierre,tasa_mora_90,desviacion_montos,coef_variacion_montos,creditos_por_cliente,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag,bloque_id
date,str,str,i64,i64,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",i64,i64,i64,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",i64,str
2021-01-01,"""2""","""1""",1,1548,16572000.00,16572000.00,53.21,12.77,317.07,0.00,0.00,0.00,0.00,0.00,0.00,0,1548,34,0.00,100.00,0.00,5648.49,0.03,45.53,2480.00,4503.33,0,"""2_1_1"""
2021-01-01,"""2""","""1""",2,2571,27488800.00,27488800.00,58.64,12.77,563.92,0.00,0.00,0.00,0.00,0.00,0.00,5,2571,65,0.19,100.00,0.00,7963.60,0.03,39.55,14183.33,30443.11,0,"""2_1_2"""
2021-01-01,"""2""","""1""",3,1590,18014960.00,18014960.00,48.31,12.77,739.66,0.00,0.00,0.00,0.00,0.00,0.00,0,1590,48,0.00,100.00,0.00,11802.11,0.07,33.13,-38.16,-34.46,0,"""2_1_3"""
2021-01-01,"""2""","""1""",4,2933,20973920.00,20973920.00,54.61,12.77,48.45,0.00,0.00,0.00,0.00,0.00,0.00,0,2933,77,0.00,100.00,0.00,5833.44,0.03,38.09,84.47,16.43,0,"""2_1_4"""
2021-01-01,"""2""","""1""",5,306,3920449.60,3920449.60,39.06,12.77,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0,306,12,0.00,100.00,0.00,5505.79,0.14,25.50,-89.57,-81.31,0,"""2_1_5"""


## Poblar dimensiones

Se extraen los valores únicos de la vista materializada y se insertan de forma idempotente
(sin duplicados) usando `ON CONFLICT DO NOTHING`.

In [56]:
cursor = conn.cursor()
for nombre, sql in [
    ("dim_tiempo", SQL_INSERT_DIM_TIEMPO),
    ("dim_riesgo", SQL_INSERT_DIM_RIESGO),
    ("dim_sector", SQL_INSERT_DIM_SECTOR),
    ("dim_sucursal", SQL_INSERT_DIM_SUCURSAL),
]:
    cursor.execute(sql)
    print(f"  {nombre}: {cursor.rowcount} filas insertadas")
cursor.close()

print("\nDimensiones pobladas.")

  dim_tiempo: 0 filas insertadas
  dim_riesgo: 0 filas insertadas
  dim_sector: 0 filas insertadas
  dim_sucursal: 64 filas insertadas

Dimensiones pobladas.


In [57]:
for tabla in ["dim_tiempo", "dim_riesgo", "dim_sector", "dim_sucursal"]:
    df = pl.read_database(f"SELECT COUNT(*) AS total FROM {tabla}", conn)
    print(f"  {tabla}: {df['total'][0]:,} registros")

  dim_tiempo: 36 registros
  dim_riesgo: 1 registros
  dim_sector: 16 registros
  dim_sucursal: 64 registros


## Poblar tabla de hechos `fact_creditos_mensual`

Se hace JOIN de la vista materializada con las 4 dimensiones para resolver las FKs,
y se inserta con upsert para ser idempotente.

In [58]:
cursor = conn.cursor()
print("Poblando fact_creditos_mensual...")
cursor.execute(SQL_UPSERT_FACT_CREDITOS)
print(f"fact_creditos_mensual: {cursor.rowcount} filas insertadas/actualizadas")
cursor.close()

Poblando fact_creditos_mensual...
fact_creditos_mensual: 5562 filas insertadas/actualizadas


## Validación

In [59]:
print("=" * 60)
print("VALIDACIÓN DEL DATAMART")
print("=" * 60)

# 1. Conteo por tabla
print("\n1) Conteo de registros:")
for tabla in ["mv_creditos_mensuales", "dim_tiempo", "dim_riesgo",
              "dim_sector", "dim_sucursal", "fact_creditos_mensual"]:
    df = pl.read_database(f"SELECT COUNT(*) AS total FROM {tabla}", conn)
    print(f"   {tabla}: {df['total'][0]:,}")

# 2. Distribución de crisis_flag
print("\n2) Distribución de crisis_flag:")
df_crisis = pl.read_database(
    "SELECT crisis_flag, COUNT(*) AS cantidad FROM fact_creditos_mensual GROUP BY crisis_flag ORDER BY crisis_flag",
    conn
)
print(df_crisis)

# 3. Verificar FKs nulas
print("\n3) Verificación de FKs nulas (debe ser 0):")
df_nulls = pl.read_database(
    "SELECT COUNT(*) AS nulos FROM fact_creditos_mensual WHERE id_tiempo IS NULL OR id_riesgo IS NULL OR id_sector IS NULL OR id_sucursal IS NULL",
    conn
)
print(f"   Filas con FK nula: {df_nulls['nulos'][0]}")

# 4. Verificar duplicados
print("\n4) Verificación de duplicados (debe ser 0):")
df_dups = pl.read_database(
    "SELECT COUNT(*) AS duplicados FROM (SELECT id_tiempo, id_riesgo, id_sector, id_sucursal FROM fact_creditos_mensual GROUP BY id_tiempo, id_riesgo, id_sector, id_sucursal HAVING COUNT(*) > 1) t",
    conn
)
print(f"   Filas duplicadas: {df_dups['duplicados'][0]}")

# 5. Rango de fechas
print("\n5) Rango de fechas en dim_tiempo:")
df_rango = pl.read_database(
    "SELECT MIN(mes) AS inicio, MAX(mes) AS fin FROM dim_tiempo",
    conn
)
print(f"   Desde: {df_rango['inicio'][0]}  Hasta: {df_rango['fin'][0]}")

print("\n" + "=" * 60)

VALIDACIÓN DEL DATAMART

1) Conteo de registros:
   mv_creditos_mensuales: 5,562
   dim_tiempo: 36
   dim_riesgo: 1
   dim_sector: 16
   dim_sucursal: 64
   fact_creditos_mensual: 5,562

2) Distribución de crisis_flag:
shape: (2, 2)
┌─────────────┬──────────┐
│ crisis_flag ┆ cantidad │
│ ---         ┆ ---      │
│ i64         ┆ i64      │
╞═════════════╪══════════╡
│ 0           ┆ 4508     │
│ 1           ┆ 1054     │
└─────────────┴──────────┘

3) Verificación de FKs nulas (debe ser 0):
   Filas con FK nula: 0

4) Verificación de duplicados (debe ser 0):
   Filas duplicadas: 0

5) Rango de fechas en dim_tiempo:
   Desde: 2019-01-01  Hasta: 2021-12-01



In [60]:
print("Muestra de fact_creditos_mensual (primeras 10 filas):")
df_fact_sample = pl.read_database(
    """
    SELECT f.*, dt.mes, dr.codigo_riesgo, ds.codigo_sector
    FROM fact_creditos_mensual f
    JOIN dim_tiempo dt ON f.id_tiempo = dt.id_tiempo
    JOIN dim_riesgo dr ON f.id_riesgo = dr.id_riesgo
    JOIN dim_sector ds ON f.id_sector = ds.id_sector
    ORDER BY dt.mes
    LIMIT 10
    """,
    conn
)
df_fact_sample

Muestra de fact_creditos_mensual (primeras 10 filas):


id_tiempo,id_riesgo,id_sector,id_sucursal,num_creditos,monto_total,monto_promedio,tot_dias_mora_promedio,tot_num_moras_promedio,tasa_mora_90,tasa_judicial,tasa_cierre,total_gestion_cobro,total_costo_judicial,tasa_interes_promedio,saldo_promedio,creditos_cerrados,num_clientes_unicos,creditos_por_cliente,plazo_promedio,desviacion_montos,coef_variacion_montos,tasa_crecimiento_creditos,tasa_crecimiento_monto,crisis_flag,bloque_id,mes,codigo_riesgo,codigo_sector
i64,i64,i64,i64,i64,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",i64,i64,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]",i64,str,date,str,str
21,1,2,1,226,2658200.00,2658200.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,2459.10,226,9,25.11,50.74,7342.52,0.28,-65.86,-79.76,0,"""2_1_49""",2021-01-01,"""2""","""1"""
21,1,2,2,58,290000.00,290000.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,0.00,58,1,58.00,60.00,0.00,0.00,null,null,0,"""2_1_71""",2021-01-01,"""2""","""1"""
21,1,2,3,358,4358000.00,4358000.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,1768.68,358,6,59.67,67.11,6664.69,0.15,49.17,41.81,0,"""2_1_31""",2021-01-01,"""2""","""1"""
21,1,2,4,474,9969000.00,9969000.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,3434.33,474,8,59.25,67.24,9098.77,0.09,404.26,1206.47,0,"""2_1_43""",2021-01-01,"""2""","""1"""
21,1,2,5,1630,19557980.00,19557980.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,0.00,1630,36,45.28,58.20,11740.45,0.06,166.34,159.42,0,"""2_1_7""",2021-01-01,"""2""","""1"""
21,1,2,6,406,3138000.00,3138000.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,0.00,406,9,45.11,50.07,3918.99,0.12,-33.00,-55.88,0,"""2_1_19""",2021-01-01,"""2""","""1"""
21,1,2,7,1314,13433000.00,13433000.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,27.28,1314,36,36.50,67.36,7284.43,0.05,267.04,208.24,0,"""2_1_32""",2021-01-01,"""2""","""1"""
21,1,2,8,1590,18014960.00,18014960.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,739.66,1590,48,33.13,48.31,11802.11,0.07,-38.16,-34.46,0,"""2_1_3""",2021-01-01,"""2""","""1"""
21,1,2,9,556,9766700.00,9766700.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,12.77,0.00,556,18,30.89,42.63,19865.56,0.20,479.17,1096.90,0,"""2_1_51""",2021-01-01,"""2""","""1"""


## Consultas de ejemplo para Superset

Estas consultas sirven como base para los dashboards.

In [61]:
print("Top 10 bloques con mayor tasa_judicial promedio:")
df_top = pl.read_database(
    """
    SELECT bloque_id,
           ROUND(AVG(tasa_judicial), 2) AS tasa_judicial_prom,
           ROUND(AVG(tasa_mora_90), 2) AS tasa_mora_90_prom,
           SUM(num_creditos) AS total_creditos,
           SUM(crisis_flag) AS meses_crisis
    FROM fact_creditos_mensual
    GROUP BY bloque_id
    ORDER BY tasa_judicial_prom DESC
    LIMIT 10
    """,
    conn
)
df_top

Top 10 bloques con mayor tasa_judicial promedio:


bloque_id,tasa_judicial_prom,tasa_mora_90_prom,total_creditos,meses_crisis
str,"decimal[38,2]","decimal[38,2]",i64,i64
"""2_13_8""",100.00,0.00,37,1
"""2_35_56""",100.00,0.00,48,1
"""2_43_54""",100.00,0.00,49,1
"""2_43_70""",100.00,0.00,48,1
"""2_46_48""",100.00,0.00,120,1
"""2_35_41""",100.00,0.00,72,2
"""2_44_41""",68.14,0.00,928,7
"""2_43_48""",61.06,0.00,987,6
"""2_44_37""",58.85,0.00,7022,12


In [62]:
print("Evolución mensual de tasa_mora_90 por sector:")
df_evol = pl.read_database(
    """
    SELECT dt.mes, ds.codigo_sector,
           ROUND(AVG(f.tasa_mora_90), 2) AS tasa_mora_90_prom,
           SUM(f.num_creditos) AS num_creditos
    FROM fact_creditos_mensual f
    JOIN dim_tiempo dt ON f.id_tiempo = dt.id_tiempo
    JOIN dim_sector ds ON f.id_sector = ds.id_sector
    GROUP BY dt.mes, ds.codigo_sector
    ORDER BY dt.mes, ds.codigo_sector
    LIMIT 30
    """,
    conn
)
df_evol

Evolución mensual de tasa_mora_90 por sector:


mes,codigo_sector,tasa_mora_90_prom,num_creditos
date,str,"decimal[38,2]",i64
2021-01-01,"""1""",0.00,37758
2021-01-01,"""13""",0.00,94
2021-01-01,"""2""",0.00,741
2021-01-01,"""33""",0.00,172
2021-01-01,"""34""",0.00,5651
…,…,…,…
2021-03-01,"""1""",0.00,66748
2021-03-01,"""13""",0.00,111
2021-03-01,"""2""",0.00,1856


## Refresco incremental

Para refrescar el datamart con datos nuevos, ejecutar estas celdas.
La vista materializada se recrea (o se usa `REFRESH` si ya tiene índice único).
Las dimensiones y la fact table se actualizan con upsert.

In [63]:
cursor = conn.cursor()

cursor.execute("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_creditos_mensuales")
print("MV refrescada.")

for sentencia_sql in [SQL_INSERT_DIM_TIEMPO, SQL_INSERT_DIM_RIESGO, SQL_INSERT_DIM_SECTOR, SQL_INSERT_DIM_SUCURSAL]:
    cursor.execute(sentencia_sql)
print("Dimensiones actualizadas.")

cursor.execute(SQL_UPSERT_FACT_CREDITOS)
print(f"Fact table: {cursor.rowcount} filas actualizadas.")

cursor.close()

MV refrescada.
Dimensiones actualizadas.
Fact table: 5562 filas actualizadas.


![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*